# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Card status:** ML-05 was retired upstream on 2026-07-13 and its core folded into ML-04. This is
the optional stretch version.

**Written after ML-09, and it does not pretend otherwise.** A Week-3 notebook claiming to have caught
the leak that ML-09 actually caught would be the exact dishonesty this repo has spent six cards
arguing against. So this notebook does something more useful than a reconstruction: it documents the
feature vector properly, and then runs **both** leakage hunts side by side, the one a careful person
runs in Week 3 and the one that actually works, so the difference between them is on the record.

The short version of that difference: the Week-3 hunt checks features **one at a time**. The leak in
this corpus lives in a **ratio between two of them**. No amount of care at the single-feature level
finds it.

Skill loaded: `skills/hunting-leakage-and-validating`.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The vector is built by the reference pipeline's `scripts/01_prepare_features.py`, which takes the
44-column raw slice to 52 columns. Per `work/README.md` rule 1 the reference pipeline stays pristine,
so this notebook **runs it and audits the result** rather than reimplementing it.

The eight engineered columns fall into three groups:

| Group | Columns | Why they exist |
|---|---|---|
| **Log transforms** | `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d` | Every traffic column is heavy-tailed (ML-06 measured skew up to 27). Linear models need the tails compressed; without this the giants decide every coefficient. |
| **Presence flags** | `has_clicks`, `has_ai_sessions` | A zero in these columns is ambiguous between "none" and "not measured". An explicit flag lets a model separate the two instead of inferring it. |
| **Composites** | `measurable_opportunity`, `age_tier_order` | `measurable_opportunity` marks rows with enough traffic to diagnose anything; `age_tier_order` gives the age tiers a usable ordering. |

Plus five tier encodings (`age_tier`, `freshness_tier`, `word_count_tier`, `char_count_tier`,
`impression_tier`, `position_tier`) that bucket continuous fields into the bands a human reads.

In [1]:
# Section 1 - build the vector via the reference pipeline, then audit what came out.
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)


def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents, Path("/content/FlyRank-Machine-Learning-Internship")]:
        if (base / "work" / "outputs" / "data_contract.json").exists():
            return base
    raise FileNotFoundError(
        "Could not locate the repo root. In Colab, clone the repo first:\n"
        "  !git clone https://github.com/Dawngend/FlyRank-Machine-Learning-Internship.git"
    )


ROOT = find_repo_root()
RAW = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
VECTOR = ROOT / "data" / "processed" / "refresh_feature_vector.csv"

if not VECTOR.exists():
    print("feature vector missing, running the reference prepare step...")
    subprocess.run([sys.executable, str(ROOT / "scripts" / "01_prepare_features.py")], check=True)

raw = pd.read_csv(RAW)
frame = pd.read_csv(VECTOR)

print(f"raw    : {raw.shape[0]:,} rows x {raw.shape[1]} columns")
print(f"vector : {frame.shape[0]:,} rows x {frame.shape[1]} columns")
assert len(raw) == len(frame), "the prepare step must not add or drop rows"

engineered = [c for c in frame.columns if c not in raw.columns]
print(f"\nengineered columns ({len(engineered)}):")
for col in engineered:
    kind = ("log transform" if col.startswith("log_")
            else "presence flag" if col.startswith("has_")
            else "label" if "label" in col
            else "composite")
    print(f"  {col:<26} {kind}")

print("\n--- the log transforms do what they are for ---")
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    log_col = f"log_{col}"
    if log_col in frame.columns:
        print(f"  {col:<20} skew {frame[col].skew():>7.2f}  ->  "
              f"{log_col:<24} skew {frame[log_col].skew():>6.2f}")

raw    : 30,000 rows x 44 columns
vector : 30,000 rows x 52 columns

engineered columns (8):
  is_declining_label         label
  log_impressions_90d        log transform
  log_clicks_90d             log transform
  log_sessions_90d           log transform
  log_ai_sessions_90d        log transform
  has_clicks                 presence flag
  has_ai_sessions            presence flag
  measurable_opportunity     composite

--- the log transforms do what they are for ---
  impressions_90d      skew   11.38  ->  log_impressions_90d      skew  -0.39
  clicks_90d           skew   18.35  ->  log_clicks_90d           skew   1.21
  sessions_90d         skew   12.13  ->  log_sessions_90d         skew   0.71
  ai_sessions_90d      skew   19.14  ->  log_ai_sessions_90d      skew   5.04


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The third question is the one that decides whether a feature is legal. **"Available-when"** asks: at
the moment the decision is made, does this value already exist? A feature that is only knowable after
the outcome is not a feature, it is the outcome.

Three categories, and the cell below classifies all 52 columns into them:

- **PRE-DECISION** — describes the page's state and its trailing 90-day window. Known at decision
  time. Legal.
- **LABEL WINDOW** — describes the 30-day comparison the label is computed from. Known only when the
  label is known. **Illegal.**
- **NOT A FEATURE** — identifiers and out-of-scope fields.

### The missingness note that matters

`word_count` is zero on 25.7% of rows, and a published page cannot have zero words, so that zero
means **not measured** rather than **none**. ML-06 traced the consequence: the `<1000` word-count
tier has a median of 4 impressions and a declining rate of 0.207 against a base of 0.542, because it
is not a population of thin pages at all, it is the unmeasured rows.

A blind `fillna(0)` would have injected that measurement artefact into the model as if it were a
content signal. The presence flags exist so the distinction survives into the feature vector instead
of being flattened.

In [2]:
# Section 2 - classify every column by when it becomes available.
LABEL_WINDOW = {
    "trend_direction", "trend_pct", "is_declining_label",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
NOT_A_FEATURE = {"content_id", "client_id", "provider_used", "model_used"}


def availability(col: str) -> str:
    if col in LABEL_WINDOW:
        return "LABEL WINDOW (illegal)"
    if col in NOT_A_FEATURE:
        return "NOT A FEATURE"
    return "PRE-DECISION (legal)"


notes = pd.DataFrame({
    "column": frame.columns,
    "dtype": [str(frame[c].dtype) for c in frame.columns],
    "available_when": [availability(c) for c in frame.columns],
    "missing_or_zero": [round(float((frame[c] == 0).mean()), 4)
                        if pd.api.types.is_numeric_dtype(frame[c])
                        else round(float(frame[c].isna().mean()), 4) for c in frame.columns],
    "distinct": [int(frame[c].nunique()) for c in frame.columns],
})
print("--- availability audit, all columns ---")
print(notes["available_when"].value_counts().to_string())
print()
print(notes.sort_values(["available_when", "column"]).to_string(index=False))

print("\n--- categorical handling ---")
cats = [c for c in frame.columns if frame[c].dtype == object and c not in NOT_A_FEATURE]
for col in cats:
    print(f"  {col:<20} {frame[col].nunique():>3} levels: "
          f"{', '.join(map(str, frame[col].unique()[:6]))}")

print("\n--- the missingness that is not really zero ---")
for col in ["word_count", "char_count", "search_volume", "clicks_90d", "ai_sessions_90d"]:
    zero_share = (frame[col] == 0).mean()
    reading = "likely NOT MEASURED" if col in ("word_count", "char_count") else "plausibly a true zero"
    print(f"  {col:<20} {zero_share:>6.1%} zeros   -> {reading}")

thin = frame[frame["word_count_tier"] == "<1000"]
print(f"\nthe '<1000' word tier: n={len(thin):,}, median impressions "
      f"{thin['impressions_90d'].median():.0f}, declining rate {thin['is_declining_label'].mean():.3f} "
      f"(base {frame['is_declining_label'].mean():.3f})")
print("-> not a population of thin pages; it is where length was never recorded.")

--- availability audit, all columns ---
available_when
PRE-DECISION (legal)      39
LABEL WINDOW (illegal)     9
NOT A FEATURE              4

                column   dtype         available_when  missing_or_zero  distinct
       clicks_last_30d   int64 LABEL WINDOW (illegal)           0.6122       239
       clicks_prev_30d   int64 LABEL WINDOW (illegal)           0.6080       258
  impressions_last_30d   int64 LABEL WINDOW (illegal)           0.0849      5182
  impressions_prev_30d   int64 LABEL WINDOW (illegal)           0.1129      5931
    is_declining_label   int64 LABEL WINDOW (illegal)           0.4579         2
     sessions_last_30d   int64 LABEL WINDOW (illegal)           0.2528       359
     sessions_prev_30d   int64 LABEL WINDOW (illegal)           0.2243       311
       trend_direction  object LABEL WINDOW (illegal)           0.0000         5
             trend_pct float64 LABEL WINDOW (illegal)           0.1277      2712
             client_id  object          NOT A F

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Two hunts, run side by side. **Hunt A** is what a careful person does in Week 3. **Hunt B** is what
actually finds the leak in this corpus.

### Hunt A — score every feature alone against the label

The standard move: if a single column separates the label suspiciously well, it is probably derived
from it. Flag anything at ROC-AUC 0.90 or above.

**Result: one flag, `trend_pct`, which is the label's own source column and was already excluded by
name.** Every column that is actually in the model sits between 0.41 and 0.59, comfortably close to
chance. Hunt A reports the feature set clean.

Hunt A is not sloppy. It is a real test, correctly run, and it is **wrong**.

### Hunt B — rebuild the label from its own definition

Instead of asking "does any feature correlate with the label", ask: **"the label is defined by an
equation. Which columns appear in that equation, and did any of them ship?"**

The label is `trend_direction == "down"`, and `trend_direction` is the 30-day-over-previous-30-day
impression change. The feature vector ships `impressions_last_30d` and `impressions_prev_30d`. Those
are not correlated with the label. They *are* the label, before it was reduced to a boolean.

**The reconstruction is exact: 30,000 of 30,000 rows.**

### Why Hunt A could never have found it

| Column | ROC-AUC alone |
|---|---:|
| `impressions_last_30d` | 0.486 (worse than a coin flip) |
| `impressions_prev_30d` | 0.621 |
| **their ratio** | **1.000** |

Neither column is remotely suspicious on its own. Their ratio is a perfect separator. **A
one-feature-at-a-time scan is structurally incapable of seeing a leak that lives in an
interaction**, and no amount of lowering the threshold fixes that, it only produces false positives
on the legitimate features.

### The threshold discrepancy, found the same way

Recovering the decision boundary empirically from `trend_direction` puts the cut at ±20%: `down` tops
out at −20.021%, `stable` spans exactly [−20, +20], `up` starts at +20.025%. The published FlyRank
paper documents ±10% (p. 5). At the documented threshold the reconstruction agrees on 93.3% of rows;
at ±20%, on 100.0%.

Recorded rather than silently corrected, because a reader following the documented definition builds
a different label and will not reproduce anything in this repo.

### Privacy

Separate from leakage and quickly checked: the only client-linked column is the pseudonymous
`client_id`, used for fold grouping and never as a feature. No URLs, page titles, query strings, or
free text of any kind ship in this vector. The cell below asserts it.

In [3]:
# Section 3 - both hunts, side by side.
from sklearn.metrics import roc_auc_score  # noqa: E402

y = frame["is_declining_label"].to_numpy()
sys.path.insert(0, str(ROOT / "work" / "scripts"))
from train_refresh_model import CATEGORICAL, NUMERIC  # noqa: E402

MODEL_FEATURES = set(NUMERIC + CATEGORICAL)

# --- Hunt A: one feature at a time -------------------------------------------
print("--- HUNT A: every numeric column scored alone ---")
scores = []
for col in frame.columns:
    if col in {"is_declining_label", "content_id"} or not pd.api.types.is_numeric_dtype(frame[col]):
        continue
    values = frame[col].to_numpy(dtype=float)
    usable = np.isfinite(values)
    if usable.sum() < 100 or len(np.unique(values[usable])) < 2:
        continue
    auc = float(roc_auc_score(y[usable], values[usable]))
    scores.append({"feature": col, "auc": round(auc, 4),
                   "separation": round(abs(auc - 0.5), 4),
                   "in_model": col in MODEL_FEATURES})
scores = pd.DataFrame(scores).sort_values("separation", ascending=False)
flagged = scores[scores["separation"] >= 0.40]
print(f"columns flagged at AUC >= 0.90: {len(flagged)}")
print(flagged.to_string(index=False))
in_model = scores[scores["in_model"]]
print(f"\nevery column actually in the model: AUC {in_model['auc'].min():.3f} to "
      f"{in_model['auc'].max():.3f}")
print("HUNT A VERDICT: feature set clean. (Correct test. Wrong answer.)")

# --- Hunt B: rebuild the label ------------------------------------------------
print("\n--- HUNT B: rebuild the label from its definition ---")
last = frame["impressions_last_30d"].to_numpy(dtype=float)
prev = frame["impressions_prev_30d"].to_numpy(dtype=float)
with np.errstate(divide="ignore", invalid="ignore"):
    pct = np.where(prev > 0, (last - prev) / prev * 100.0, np.nan)

print("where each documented direction actually sits on the recomputed change:")
for direction in ("down", "stable", "up"):
    band = pct[(frame["trend_direction"] == direction).to_numpy() & ~np.isnan(pct)]
    print(f"  {direction:<7} n={band.size:>6,}  {band.min():>10.3f}% to {band.max():>10.3f}%")

for threshold in (-10.0, -20.0):
    rebuilt = np.where(prev > 0, pct < threshold, False)
    print(f"  reconstruction at {threshold:>6.1f}%: agrees on {(rebuilt == y).mean():.6f} of rows "
          f"({int((rebuilt == y).sum()):,} / {len(frame):,})")

rebuilt = np.where(prev > 0, pct < -20.0, False)
assert (rebuilt == y).all(), "the recovered rule should reconstruct the label exactly"
print("HUNT B VERDICT: exact label reconstruction from two shipped columns.")

# --- why A could not find what B found ---------------------------------------
print("\n--- why Hunt A could not have found it ---")
for col in ["impressions_last_30d", "impressions_prev_30d"]:
    print(f"  {col:<24} alone : AUC {roc_auc_score(y, frame[col]):.4f}")
with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.where(prev > 0, last / prev, np.nan)
usable = ~np.isnan(ratio)
print(f"  {'their ratio':<24}       : AUC {roc_auc_score(y[usable], -ratio[usable]):.4f}")
print("-> a single-feature scan cannot see a leak that lives in an interaction.")

# --- privacy ------------------------------------------------------------------
print("\n--- privacy check ---")
text_like = [c for c in frame.columns
             if frame[c].dtype == object and frame[c].astype(str).str.len().max() > 40]
assert not text_like, f"free-text or URL-like columns present: {text_like}"
identifying = [c for c in frame.columns
               if any(k in c.lower() for k in ("url", "title", "query", "email", "name", "path"))]
assert not identifying, f"potentially identifying columns present: {identifying}"
assert "client_id" not in MODEL_FEATURES, "client_id must be grouping only, never a feature"
print(f"[OK] no free-text, URL, title, or query columns in {len(frame.columns)} columns")
print(f"[OK] client_id used for grouping only; {frame['client_id'].nunique()} pseudonymous values")
print(f"[OK] longest object value is {max(frame[c].astype(str).str.len().max() for c in frame.columns if frame[c].dtype == object)} chars (tier labels)")

--- HUNT A: every numeric column scored alone ---


columns flagged at AUC >= 0.90: 1
  feature  auc  separation  in_model
trend_pct  0.0         0.5     False

every column actually in the model: AUC 0.408 to 0.585
HUNT A VERDICT: feature set clean. (Correct test. Wrong answer.)

--- HUNT B: rebuild the label from its definition ---
where each documented direction actually sits on the recomputed change:
  down    n=16,262    -100.000% to    -20.021%
  stable  n= 5,962     -20.000% to     20.000%
  up      n= 4,388      20.025% to  44900.000%
  reconstruction at  -10.0%: agrees on 0.933433 of rows (28,003 / 30,000)
  reconstruction at  -20.0%: agrees on 1.000000 of rows (30,000 / 30,000)
HUNT B VERDICT: exact label reconstruction from two shipped columns.

--- why Hunt A could not have found it ---
  impressions_last_30d     alone : AUC 0.4857
  impressions_prev_30d     alone : AUC 0.6214
  their ratio                    : AUC 1.0000
-> a single-feature scan cannot see a leak that lives in an interaction.

--- privacy check ---
[OK] no 

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Field | Excluded by | Why |
|---|---|---|
| `is_declining_label` | ML-04 contract | The label itself. |
| `trend_direction` | ML-04 contract | The label is `trend_direction == "down"`. |
| `trend_pct` | ML-04 contract | The magnitude behind the same label. The only column Hunt A flags. |
| `impressions_last_30d` | **ML-09 audit** | With the row below, reconstructs the label exactly. Invisible to Hunt A at AUC 0.486. |
| `impressions_prev_30d` | **ML-09 audit** | The other half of that ratio. AUC 0.621 alone. |
| `clicks_last_30d`, `clicks_prev_30d` | availability | Same label window. Not used in the label's definition, but knowable only when the label is. |
| `sessions_last_30d`, `sessions_prev_30d` | availability | Same reason. |
| `provider_used`, `model_used` | ML-04 contract | Which AI wrote the draft is out of scope for "which page does a human review first", and modelling it invites a conclusion the data cannot support. |
| `content_id` | identifier | A row name, not a signal. |
| `client_id` | design | **Grouping only.** As a feature it lets a model predict a client's base rate instead of reading a page; ML-09 measured that shortcut at +22.7% inflated precision@50. |

**Two of these ten exclusions were found rather than planned**, and both by the same method: starting
from the label's definition and asking which shipped columns appear in it. That is an audit of the
data dictionary rather than of the correlation matrix, and it is the part of this notebook worth
carrying to the next project.

The cell below asserts every exclusion against the live feature list, so a future edit that
reintroduces one fails loudly instead of quietly.

In [4]:
# Section 4 - the exclusions, asserted rather than listed.
EXCLUSIONS = {
    "is_declining_label": "the label itself",
    "trend_direction": "the label is trend_direction == 'down'",
    "trend_pct": "the magnitude behind the same label",
    "impressions_last_30d": "reconstructs the label exactly, with impressions_prev_30d",
    "impressions_prev_30d": "reconstructs the label exactly, with impressions_last_30d",
    "clicks_last_30d": "label window; knowable only when the label is",
    "clicks_prev_30d": "label window; knowable only when the label is",
    "sessions_last_30d": "label window; knowable only when the label is",
    "sessions_prev_30d": "label window; knowable only when the label is",
    "provider_used": "out of scope per the ML-04 contract",
    "model_used": "out of scope per the ML-04 contract",
    "content_id": "a row name, not a signal",
    "client_id": "grouping only; as a feature it is a client lookup table",
}

print(f"model feature set: {len(MODEL_FEATURES)} columns "
      f"({len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical)")
print(f"excluded         : {len(EXCLUSIONS)} columns\n")

violations = [c for c in EXCLUSIONS if c in MODEL_FEATURES]
for col, why in EXCLUSIONS.items():
    present = col in frame.columns
    status = "VIOLATION" if col in MODEL_FEATURES else "excluded "
    print(f"  {status} {col:<24} {why}" + ("" if present else "   [not in this vector]"))
assert not violations, f"excluded columns present in the model feature set: {violations}"

covered = MODEL_FEATURES | set(EXCLUSIONS)
unaccounted = [c for c in frame.columns if c not in covered]
print(f"\ncolumns accounted for: {len(covered & set(frame.columns))} of {len(frame.columns)}")
if unaccounted:
    print(f"not in the model and not on the exclusion list ({len(unaccounted)}):")
    for col in unaccounted:
        print(f"  {col:<26} {availability(col)}")
    print("-> these are derived or tier columns superseded by the encodings the model does use;")
    print("   none of them is in the LABEL WINDOW set, which is the condition that matters.")
    leaky = [c for c in unaccounted if c in LABEL_WINDOW]
    assert not leaky, f"unaccounted columns from the label window: {leaky}"

print("\n[OK] no excluded column reaches the model, and nothing unaccounted for comes from the")
print("     label window.")

model feature set: 26 columns (18 numeric, 8 categorical)
excluded         : 13 columns

  excluded  is_declining_label       the label itself
  excluded  trend_direction          the label is trend_direction == 'down'
  excluded  trend_pct                the magnitude behind the same label
  excluded  impressions_last_30d     reconstructs the label exactly, with impressions_prev_30d
  excluded  impressions_prev_30d     reconstructs the label exactly, with impressions_last_30d
  excluded  clicks_last_30d          label window; knowable only when the label is
  excluded  clicks_prev_30d          label window; knowable only when the label is
  excluded  sessions_last_30d        label window; knowable only when the label is
  excluded  sessions_prev_30d        label window; knowable only when the label is
  excluded  provider_used            out of scope per the ML-04 contract
  excluded  model_used               out of scope per the ML-04 contract
  excluded  content_id               a r

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — asserted in section 3
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**The one line to carry forward:** a single-feature leakage scan cannot see a leak that lives in an
interaction. Audit the label's definition against the data dictionary, not just the correlation
table.